In [0]:
%sql
use catalog trueanalytics_data;

In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T 
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

In [0]:
# parameter: par_month
dbutils.widgets.text("par_month", "202510")
par_month = dbutils.widgets.get("par_month")

try:
  par_month = int(par_month)
except ValueError:
  par_month = 0
  raise ValueError("par_month value must be numeric")

if par_month!=0:
  pass
else:
  dbutils.notebook.exit("Aborting as ondition not met. Further tasks will be skipped")

# customer360 date
par_month_obj = datetime.strptime(str(par_month), '%Y%m')
next_par_month_obj = par_month_obj + relativedelta(months=1)
cust360_date = int(next_par_month_obj.strftime('%Y%m') + '01')

# debug
display('par_month = ',par_month)
display(' customer360_date = ',cust360_date)


In [0]:
# master data
prep_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_footfall.parquet'
prep_freq_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_flag_freq.parquet'
prep_feature_360 = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_360_feature.parquet'
profile_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/profile_chula.csv'
date_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/day_type_apr_june_26.csv'
nantional_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/country_group_chula.csv'
home_region_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/home_region_chula.csv'

# report path
report_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/report/report1/{par_month}/'

In [0]:
df_date = (spark.read
      .option("header", "true").option("inferSchema", "true")
      .csv(date_path)).select('date','WEEK','day_type_final')

In [0]:
df = spark.read.parquet(prep_path)
     # raw footfall by mall & customertype (is_bmr, is_non_bmr, is_foriegner)

df_freq = spark.read.parquet(prep_freq_path) # raw freq by mall
df_360 = spark.read.parquet(prep_feature_360)
     # raw profile
df_merge = df.join(F.broadcast(df_freq), ['msisdn','name'], 'left')\
    .join(F.broadcast(df_360.drop('is_bmr','is_non_bmr','is_foriegner','a_country_name','demo_tourist_sim_v1_tourist_bin','roaming_flag')), ['msisdn'], 'inner')
df_merge = df_merge\
    .join(df_date, [df_merge.par_day == df_date.date], "left")\
    .withColumnRenamed('par_month','month')
display(df_merge.count())

In [0]:
df_merge = df_merge.withColumn('dwelling_time_hr', F.when(F.col('actual_total_duration_day').between(900, 3600), F.lit('more_than_15_<1hr'))
  .when(F.col('actual_total_duration_day').between(3600, 7200), F.lit('1-2'))
  .when(F.col('actual_total_duration_day').between(7201, 10800), F.lit('2-3'))
  .when(F.col('actual_total_duration_day').between(10801, 14400), F.lit('3-4'))
  .when(F.col('actual_total_duration_day').between(14401, 21600), F.lit('4-6'))
  .otherwise(F.lit('>6'))
)

In [0]:
df_all_columns = df_merge.fillna(0, 
    subset=[
    'weekly_mall_visitors'
    ])\
        .withColumnRenamed('name','mall')\
            .withColumnRenamed('day_type_final','day_type')

In [0]:
columns = ['msisdn'
           , F.col('geog_resident_location_v1_district_en_cat').alias('home_district')
           , F.col('geog_resident_location_v1_sub_district_en_cat').alias('home_subdistrict')]
df_cust360 = spark.read.table('trueanalytics_data.customer360.customer360_snapshot')\
    .filter((F.col('par_day')==cust360_date) & (F.col('activated_flag')=='1'))\
    .select(columns)

In [0]:
df_all_columns = df_all_columns.drop('home_district','home_subdistrict')\
    .join(df_cust360, "msisdn", 'left')


df_all_columns = df_all_columns.withColumn("home_subdistrict", 
    F.when((F.col("home_province")=='bangkok') & (F.col("home_subdistrict")=='chantharakasem'), F.lit('chan kasem'))
    .otherwise(F.col('home_subdistrict'))
)

In [0]:
core_columns = [
    'latitude',
    'longitude',
    'mall',
    'province',
    'district',
    'sub_district',
    'day_type',
    'date']

In [0]:
df_bmr_r1 = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_non_bmr_r1 = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_foreigner_r1 = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))

# save_report
save_to_csv(df_bmr_r1, report_path+f"report1_bmr_r1_{par_month}.csv")
save_to_csv(df_non_bmr_r1, report_path+f"report1_non_bmr_r1_{par_month}.csv")
save_to_csv(df_foreigner_r1, report_path+f"report1_foreigner_r1_{par_month}.csv")

In [0]:
R1D_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr']
R1D_Non_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                         'region']
R1D_Foreigner_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                       'nationality','nationality_group','foreigner_type']


df_bmr_r1d = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns+R1D_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_non_bmr_r1d = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R1D_Non_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_foreigner_r1d = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R1D_Foreigner_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))

# save_report
save_to_csv(df_bmr_r1d, report_path+f"report1_bmr_r1d_{par_month}.csv")
save_to_csv(df_non_bmr_r1d, report_path+f"report1_non_bmr_r1d_{par_month}.csv")
save_to_csv(df_foreigner_r1d, report_path+f"report1_foreigner_r1d_{par_month}.csv")

In [0]:
R1H_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                   'home_province','home_district','home_subdistrict']
R1W_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                   'work_province','work_district','work_subdistrict']
R1HW_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                    'home_province','home_district','home_subdistrict','work_province','work_district','work_subdistrict']

province_bmr = ['bangkok','nakhonpathom','nonthaburi','samutprakan','samutsakhon','pathumthani']
filter_bmr_home_province = (F.col('is_bmr')==1) & ((F.col("work_province").isin(province_bmr)) & (~F.col("home_province").isin(province_bmr)))
filter_bmr_work_province = (F.col('is_bmr')==1) & ((F.col("home_province").isin(province_bmr)) & (~F.col("work_province").isin(province_bmr)))
df_bmr_re = df_all_columns.withColumn("home_province",
                               F.when(filter_bmr_home_province,
                                      F.lit("non_bmr")).otherwise(F.col("home_province")))\
                    .withColumn("home_district",
                                F.when(filter_bmr_home_province,
                                       F.lit("non_bmr")).otherwise(F.col("home_district")))\
                    .withColumn("home_subdistrict",
                                F.when(filter_bmr_home_province,
                                       F.lit("non_bmr")).otherwise(F.col("home_subdistrict")))\
                    .withColumn("work_province",
                               F.when(filter_bmr_work_province,
                                      F.lit("non_bmr")).otherwise(F.col("work_province")))\
                    .withColumn("work_district",
                                F.when(filter_bmr_work_province,
                                       F.lit("non_bmr")).otherwise(F.col("work_district")))\
                    .withColumn("work_subdistrict",
                                F.when(filter_bmr_work_province,
                                       F.lit("non_bmr")).otherwise(F.col("work_subdistrict")))
df_bmr_r1h = df_bmr_re.filter((F.col('is_bmr')==1))\
    .groupBy(core_columns+R1H_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_bmr_r1w = df_bmr_re.filter((F.col('is_bmr')==1))\
    .groupBy(core_columns+R1W_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_bmr_r1hw = df_bmr_re.filter((F.col('is_bmr')==1))\
    .groupBy(core_columns+R1HW_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))

# save_report
save_to_csv(df_bmr_r1h, report_path+f"report1_bmr_r1h_{par_month}.csv")
save_to_csv(df_bmr_r1w, report_path+f"report1_bmr_r1w_{par_month}.csv")
save_to_csv(df_bmr_r1hw, report_path+f"report1_bmr_r1hw_{par_month}.csv")

In [0]:
R1ALL_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr'
                   ,'interest_all','monthly_pay','place_type']

R1ALL_Non_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                         'region'
                         ,'interest_all','monthly_pay']
R1ALL_Foreigner_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                       'nationality','nationality_group','foreigner_type'
                       ,'interest_all','monthly_pay']

df_bmr_r1iall = df_all_columns.filter(F.col('is_bmr')==1)\
    .withColumn("place_type",F.when(F.col("place_type") == "other", F.lit("home_work_unidentified")).otherwise(F.col("place_type")))\
    .groupBy(core_columns+R1ALL_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_all','interest')
df_non_bmr_r1iall = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R1ALL_Non_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_all','interest')
df_foreigner_r1iall = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R1ALL_Foreigner_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_all','interest')

# save_report
save_to_csv(df_bmr_r1iall, report_path+f"report1_bmr_r1all_{par_month}.csv")
save_to_csv(df_non_bmr_r1iall, report_path+f"report1_non_bmr_r1all_{par_month}.csv")
save_to_csv(df_foreigner_r1iall, report_path+f"report1_foreigner_r1all_{par_month}.csv")

In [0]:
R1IS_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr'
                   ,'interest_IS','monthly_pay','place_type']

R1IS_Non_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                         'region'
                         ,'interest_IS','monthly_pay']
R1IS_Foreigner_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                       'nationality','nationality_group','foreigner_type'
                       ,'interest_IS','monthly_pay']

df_bmr_r1is = df_all_columns.filter(F.col('is_bmr')==1)\
    .withColumn("place_type",F.when(F.col("place_type") == "other", F.lit("home_work_unidentified")).otherwise(F.col("place_type")))\
    .groupBy(core_columns+R1IS_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_IS','interest')
df_non_bmr_r1is = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R1IS_Non_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_IS','interest')
df_foreigner_r1is = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R1IS_Foreigner_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_IS','interest')

# # save_report
save_to_csv(df_bmr_r1is, report_path+f"report1_bmr_r1is_{par_month}.csv")
save_to_csv(df_non_bmr_r1is, report_path+f"report1_non_bmr_r1is_{par_month}.csv")
save_to_csv(df_foreigner_r1is, report_path+f"report1_foreigner_r1is_{par_month}.csv")

In [0]:
R1IL_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr'
                   ,'interest_IL','monthly_pay','place_type']

R1IL_Non_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                         'region'
                         ,'interest_IL','monthly_pay']
R1IL_Foreigner_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                       'nationality','nationality_group','foreigner_type'
                       ,'interest_IL','monthly_pay']

df_bmr_r1il = df_all_columns.filter(F.col('is_bmr')==1)\
    .withColumn("place_type",F.when(F.col("place_type") == "other", F.lit("home_work_unidentified")).otherwise(F.col("place_type")))\
    .groupBy(core_columns+R1IL_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_IL','interest')
df_non_bmr_r1il = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R1IL_Non_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_IL','interest')
df_foreigner_r1il = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R1IL_Foreigner_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_IL','interest')

# # save_report
save_to_csv(df_bmr_r1il, report_path+f"report1_bmr_r1il_{par_month}.csv")
save_to_csv(df_non_bmr_r1il, report_path+f"report1_non_bmr_r1il_{par_month}.csv")
save_to_csv(df_foreigner_r1il, report_path+f"report1_foreigner_r1il_{par_month}.csv")

In [0]:
R1IF_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr'
                   ,'interest_IF','monthly_pay','place_type']

R1IF_Non_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                         'region'
                         ,'interest_IF','monthly_pay']
R1IF_Foreigner_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                       'nationality','nationality_group','foreigner_type'
                       ,'interest_IF','monthly_pay']

df_bmr_r1if = df_all_columns.filter(F.col('is_bmr')==1)\
    .withColumn("place_type",F.when(F.col("place_type") == "other", F.lit("home_work_unidentified")).otherwise(F.col("place_type")))\
    .groupBy(core_columns+R1IF_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_IF','interest')
df_non_bmr_r1if = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R1IF_Non_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_IF','interest')
df_foreigner_r1if = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R1IF_Foreigner_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_IF','interest')

# # save_report
save_to_csv(df_bmr_r1if, report_path+f"report1_bmr_r1if_{par_month}.csv")
save_to_csv(df_non_bmr_r1if, report_path+f"report1_non_bmr_r1if_{par_month}.csv")
save_to_csv(df_foreigner_r1if, report_path+f"report1_foreigner_r1if_{par_month}.csv")

In [0]:
R1I_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr'
                   ,'interest_I','monthly_pay','place_type']

R1I_Non_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                         'region'
                         ,'interest_I','monthly_pay']
R1I_Foreigner_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                       'nationality','nationality_group','foreigner_type'
                       ,'interest_I','monthly_pay']

df_bmr_r1i = df_all_columns.filter(F.col('is_bmr')==1)\
    .withColumn("place_type",F.when(F.col("place_type") == "other", F.lit("home_work_unidentified")).otherwise(F.col("place_type")))\
    .groupBy(core_columns+R1I_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_I','interest')
df_non_bmr_r1i = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R1I_Non_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_I','interest')
df_foreigner_r1i = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R1I_Foreigner_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))\
        .withColumnRenamed('interest_I','interest')

# # save_report
save_to_csv(df_bmr_r1i, report_path+f"report1_bmr_r1i_{par_month}.csv")
save_to_csv(df_non_bmr_r1i, report_path+f"report1_non_bmr_r1i_{par_month}.csv")
save_to_csv(df_foreigner_r1i, report_path+f"report1_foreigner_r1i_{par_month}.csv")

In [0]:
freq_columns = ['cpn_cbd_customers','cpn_cbd_frequent_customers','cpn_non_cbd_customers','cpn_non_cbd_frequent_customers','spw_iconics_customers','spw_iconics_frequent_customers','spw_spdscsd_customers','spw_spdscsd_frequent_customers','tcc_customers','tcc_frequent_customers','the_mall_cbd_customers','the_mall_cbd_frequent_customers','the_mall_non_cbd_customers','the_mall_non_cbd_frequent_customers','spw_group_customers','spw_group_frequent_customers','cpn_cbd_weekly_active','cpn_non_cbd_weekly_active','spw_iconics_weekly_active','spw_spdscsd_weekly_active','tcc_weekly_active','the_mall_cbd_weekly_active','the_mall_non_cbd_weekly_active','spw_group_weekly_active']

R1R_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr'
                   ,'monthly_pay','place_type']

R1R_Non_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                         'region'
                         ,'monthly_pay']
R1R_Foreigner_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                       'nationality','nationality_group','foreigner_type'
                       ,'monthly_pay']

df_bmr_r1r = df_all_columns.filter(F.col('is_bmr')==1)\
    .withColumn("place_type",F.when(F.col("place_type") == "other", F.lit("home_work_unidentified")).otherwise(F.col("place_type")))\
    .groupBy(core_columns+R1R_BMR_columns+freq_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_non_bmr_r1r = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R1R_Non_BMR_columns+freq_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_foreigner_r1if = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R1R_Foreigner_columns+freq_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))

# # save_report
save_to_csv(df_bmr_r1r, report_path+f"report1_bmr_r1r_{par_month}.csv")
save_to_csv(df_non_bmr_r1r, report_path+f"report1_non_bmr_r1r_{par_month}.csv")
save_to_csv(df_foreigner_r1if, report_path+f"report1_foreigner_r1r_{par_month}.csv")